In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load league-wide game data and filter to MIN home games
data = pd.read_csv('../../../data/league_weather_2021_2025.csv')
min_df = data[data['home_team'] == 'MIN'].copy()

# Parse game_date to extract month
min_df['game_date'] = pd.to_datetime(min_df['game_date'])
min_df['month'] = min_df['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
             7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
min_df['month_name'] = min_df['month'].map(month_map)

months = sorted(min_df['month'].unique())
month_names = [month_map.get(m, str(m)) for m in months]

print(f"Total MIN home games: {len(min_df)}")
print(f"\nGames per month:")
print(min_df.groupby('month_name').size().reindex([month_map[m] for m in months]))

In [ ]:
# Weather Distributions by Month

weather_vars = {
    'temp_f':   {'label': 'Temperature (\u00b0F)', 'color': '#d62728'},
    'pres':     {'label': 'Pressure (hPa)',        'color': '#9467bd'},
    'rhum':     {'label': 'Humidity (%)',           'color': '#1f77b4'},
    'wspd_mph': {'label': 'Wind Speed (mph)',       'color': '#2ca02c'},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (col, info) in zip(axes.flat, weather_vars.items()):
    box_data = [min_df.loc[min_df['month'] == m, col].dropna().values for m in months]
    bp = ax.boxplot(box_data, patch_artist=True, labels=month_names,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor(info['color'])
        patch.set_alpha(0.6)
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel(info['label'], fontsize=11)
    ax.set_title(info['label'], fontsize=13)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle('Weather Distributions by Month \u2014 Target Field', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Wind Direction on High Wind Days

# Define high wind as top quintile
min_df['wspd_bin'] = pd.qcut(min_df['wspd_mph'], q=5, duplicates='drop')
top_bin = min_df['wspd_bin'].cat.categories[-1]
high_wind = min_df[min_df['wspd_bin'] == top_bin]

compass = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
dir_counts = high_wind['wind_dir_bucket'].value_counts().reindex(compass, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(dir_counts.index, dir_counts.values, color='steelblue', alpha=0.75,
       edgecolor='black', linewidth=0.5)
ax.set_xlabel('Wind Direction (blowing from)', fontsize=12)
ax.set_ylabel('Number of Games', fontsize=12)
ax.set_title(f'Wind Direction on High Wind Days (Top Quintile: {top_bin}) — Target Field', fontsize=13)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

for xi, v in enumerate(dir_counts.values):
    ax.text(xi, v + 0.3, str(v), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Summary Statistics Table

summary = min_df.groupby('month').agg(
    games=('temp_f', 'size'),
    temp_mean=('temp_f', 'mean'),
    temp_std=('temp_f', 'std'),
    pres_mean=('pres', 'mean'),
    pres_std=('pres', 'std'),
    rhum_mean=('rhum', 'mean'),
    rhum_std=('rhum', 'std'),
    wspd_mean=('wspd_mph', 'mean'),
    wspd_std=('wspd_mph', 'std'),
).round(2)

summary.index = [month_map.get(m, str(m)) for m in summary.index]
summary.index.name = 'Month'
summary.columns = ['Games', 'Temp Mean (\u00b0F)', 'Temp Std',
                    'Pressure Mean (hPa)', 'Pressure Std',
                    'Humidity Mean (%)', 'Humidity Std',
                    'Wind Mean (mph)', 'Wind Std']
summary